## ANN Classifier

In [1]:
final_csv_path=r"D:\Aamir Gulzar\KSA_project2\paip_data\final_merged_without_white.csv"
model_path = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\TissueClassifier_CRC100K\models\ann_crc100k_Hoptimus.pth"

In [2]:
import os
import h5py
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

# ========== CONFIGURATION ==========
patches_root = r"D:\Aamir Gulzar\KSA_project2\paip_data\patch_data"
features_root = r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\H_Optimus_1_fivecrop_Updated"
non_white_csv = r"D:\Aamir Gulzar\KSA_project2\paip_data\final_merged_without_white.csv"

BATCH_SIZE = 64  # You can adjust this based on your GPU memory

# patch_groups_dir = r"D:\Aamir Gulzar\KSA_project2\dataset\Patch_groups_dict\conch_224_fivecrop"
output_slide_features = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Tissue_Type_Clustering_FiveCrop\hoptimus_224_fivecrop"

In [3]:
root_dir = r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\H_Optimus_1_fivecrop_Updated"
saving_path = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Tissue_Type_Clustering_FiveCrop\hoptimus_224_fivecrop"

In [4]:
import torch
from typing import Tuple, Dict, Any, List
import torch.nn.functional as F
import numpy as np
import time
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

class ANNClassifier:
    def __init__(self, input_dim=None, hidden_dim1=224, hidden_dim2=128, C=1.0, max_iter=100, verbose=True, random_state=42):
        self.C = C
        self.loss_func = torch.nn.CrossEntropyLoss()
        self.max_iter = max_iter
        self.random_state = random_state
        self.verbose = verbose
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.input_dim = input_dim  # Will be set during fit() if None
        self.hidden_dim1 = hidden_dim1
        self.hidden_dim2 = hidden_dim2
        self.model = None

    def compute_loss(self, preds, labels):
        loss = self.loss_func(preds, labels)
        wreg = 0.5 * sum((param.norm(p=2) for param in self.model.parameters()))  # L2 regularization
        return loss.mean() + (1.0 / self.C) * wreg

    def predict_proba(self, feats):
        feats = feats.to(self.device)
        with torch.no_grad():
            return torch.softmax(self.model(feats), dim=1)

    def fit(self, feats, labels):
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)

        # Set input_dim if not set
        if self.input_dim is None:
            self.input_dim = feats.shape[1]

        self.model = torch.nn.Sequential(
            torch.nn.Linear(self.input_dim, self.hidden_dim1),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(self.hidden_dim1, self.hidden_dim2),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(self.hidden_dim2, 9)  # Assuming 9 classes
        ).to(self.device)

        for layer in self.model:
            if isinstance(layer, torch.nn.Linear):
                torch.nn.init.xavier_uniform_(layer.weight)

        feats = feats.to(self.device)
        labels = labels.long().to(self.device)

        opt = torch.optim.Adam(self.model.parameters(), lr=1e-3, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5)

        min_loss = np.inf
        patience = 10
        patience_counter = 0

        for epoch in range(self.max_iter):
            opt.zero_grad()
            preds = self.model(feats)
            loss = self.compute_loss(preds, labels)
            loss.backward()
            opt.step()
            scheduler.step(loss)

            if self.verbose and epoch % 10 == 0:
                print(f"Epoch {epoch}: Loss: {loss:.4f}")

            if loss < min_loss:
                min_loss = loss
                patience_counter = 0
                best_model = self.model.state_dict()
            else:
                patience_counter += 1

            # if patience_counter >= patience:
            #     print(f"Early stopping at epoch {epoch}")
            #     break

        self.model.load_state_dict(best_model)

        if self.verbose:
            print(f"Final Loss: {min_loss:.4f}")

In [5]:
def test_ANN_probe(
    classifier: ANNClassifier,
    test_feats: torch.Tensor,
    test_labels: torch.Tensor,
    prefix: str = "ann_",
    verbose: bool = True,
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    if verbose:
        print(f"ANN Probe Evaluation: Test shape {test_feats.shape}")

    probs_all = classifier.predict_proba(test_feats).cpu().numpy()
    preds_all = np.argmax(probs_all, axis=1)
    targets_all = test_labels.cpu().numpy()
    roc_kwargs = {"multi_class": "ovo", "average": "macro"}

    # ⬇️ Calculate accuracy
    accuracy = np.mean(preds_all == targets_all)
    print(f"✅ Test Accuracy: {accuracy * 100:.2f}%")

    # ⬇️ Eval metrics
    eval_metrics = get_eval_metrics(targets_all, preds_all, probs_all, True, prefix, roc_kwargs)
    dump = {"preds_all": preds_all, "probs_all": probs_all, "targets_all": targets_all}

    # ⬇️ Confusion matrix
    conf_matrix = confusion_matrix(targets_all, preds_all)
    print("Confusion Matrix:")
    plot_confusion_matrix(conf_matrix)
    
    # ⬇️ New plots for deeper insight
    plot_classification_report(targets_all, preds_all)
    plot_roc_multiclass(targets_all, probs_all, n_classes=9)
    plot_prediction_confidence(probs_all, preds_all, targets_all)


    return eval_metrics, dump


In [6]:
# Do this in main
import pandas as pd

df= pd.read_csv(final_csv_path)

def is_non_white_patch(patch_path: str, valid_patch_names: set) -> int:
    patch_name = os.path.basename(patch_path)
    return int(patch_name in valid_patch_names)

def extract_coords(patch_name: str):
    try:
        base = os.path.splitext(patch_name)[0]
        parts = base.split('_')
        x = int([p for p in parts if p.startswith('x')][0][1:])
        y = int([p for p in parts if p.startswith('y')][0][1:])
        return x, y
    except Exception as e:
        print(f"❌ Error extracting coords from {patch_name}: {e}")
        return None, None

## Hoptimus 224 five crop patch and crc100k 224 size

In [7]:
import json

with open(r'D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\TissueClassifier_CRC100K\best_parameters\best_params_Hoptimus.json', 'r') as f:
    best_params = json.load(f)

print("✅ Loaded best_params from best_params_Hoptimus.json",best_params)

✅ Loaded best_params from best_params_Hoptimus.json {'hidden_dim1': 256, 'hidden_dim2': 128, 'C': 10.0, 'max_iter': 500}


### hoptimus features 224 five crop pipeline

In [8]:
input_dim = 1536

# Initialize classifier
model = ANNClassifier(
    input_dim=input_dim,
    hidden_dim1=best_params['hidden_dim1'],
    hidden_dim2=best_params['hidden_dim2'],
    C=best_params['C'],
    max_iter=best_params.get('max_iter', 100),
    verbose=True
)

# Build model architecture (required before loading weights)
model.model = torch.nn.Sequential(
    torch.nn.Linear(model.input_dim, model.hidden_dim1),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(model.hidden_dim1, model.hidden_dim2),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(model.hidden_dim2, 9)  # 9 classes assumed
).to(model.device)

# Load trained weights
model.model = torch.load(model_path, map_location=model.device)
model.model.to(model.device)
model.model.eval()

Sequential(
  (0): Linear(in_features=1536, out_features=256, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=256, out_features=128, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.3, inplace=False)
  (6): Linear(in_features=128, out_features=9, bias=True)
)

## New appraoch label with batches

In [9]:
import pandas as pd

# Load valid patch names
non_white_df = pd.read_csv(non_white_csv)

valid_patch_names = set(
    non_white_df['patch_name'].astype(str).str.replace('.png', '.pt', regex=False)
)

# Repeat each row 5 times
df_expanded = non_white_df.loc[non_white_df.index.repeat(5)].copy()

# Add updated_patchName column
df_expanded["updated_patchName"] = (
    df_expanded["patch_name"].str.replace(".png", "", regex=False)
    + "_" + (df_expanded.groupby(level=0).cumcount() + 1).astype(str) + ".png"
)

df_expanded

,patch_location,patch_name,patch_number,patch_x,patch_y,avg_R,avg_G,avg_B,std_R,std_G,std_B,label,updated_patchName
0,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,training_data_01_MSIH_x20480_y43752_patch00024...,24,20480,43752,218.19,221.26,229.02,44.36,39.58,29.21,0,training_data_01_MSIH_x20480_y43752_patch00024...
0,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,training_data_01_MSIH_x20480_y43752_patch00024...,24,20480,43752,218.19,221.26,229.02,44.36,39.58,29.21,0,training_data_01_MSIH_x20480_y43752_patch00024...
0,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,training_data_01_MSIH_x20480_y43752_patch00024...,24,20480,43752,218.19,221.26,229.02,44.36,39.58,29.21,0,training_data_01_MSIH_x20480_y43752_patch00024...
0,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,training_data_01_MSIH_x20480_y43752_patch00024...,24,20480,43752,218.19,221.26,229.02,44.36,39.58,29.21,0,training_data_01_MSIH_x20480_y43752_patch00024...
0,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,training_data_01_MSIH_x20480_y43752_patch00024...,24,20480,43752,218.19,221.26,229.02,44.36,39.58,29.21,0,training_data_01_MSIH_x20480_y43752_patch00024...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
271321,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,validation_data_31_nonMSIH_x48784_y16624_patch...,4822,48784,16624,222.29,210.55,219.74,33.44,39.39,32.08,0,validation_data_31_nonMSIH_x48784_y16624_patch...
271321,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,validation_data_31_nonMSIH_x48784_y16624_patch...,4822,48784,16624,222.29,210.55,219.74,33.44,39.39,32.08,0,validation_data_31_nonMSIH_x48784_y16624_patch...
271321,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,validation_data_31_nonMSIH_x48784_y16624_patch...,4822,48784,16624,222.29,210.55,219.74,33.44,39.39,32.08,0,validation_data_31_nonMSIH_x48784_y16624_patch...
271321,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,validation_data_31_nonMSIH_x48784_y16624_patch...,4822,48784,16624,222.29,210.55,219.74,33.44,39.39,32.08,0,validation_data_31_nonMSIH_x48784_y16624_patch...


## Final Pipeline

In [10]:
import os
import torch
from tqdm import tqdm

# Collect all updates here for later update
updates = []
count = 0

for slide_name in tqdm(os.listdir(features_root), desc="🔍 Processing slides"):
    slide_feature_folder = os.path.join(features_root, slide_name)
    count += 1

    print(f"{count}: Processing slide: {slide_name}")

    if not os.path.isdir(slide_feature_folder):
        continue

    pt_files = sorted([
        f for f in os.listdir(slide_feature_folder)
        if f.endswith(".pt")
    ])

    batch_features = []
    batch_patch_names = []

    for idx, pt_file in enumerate(pt_files):
        pt_path = os.path.join(slide_feature_folder, pt_file)

        if is_non_white_patch(pt_path, valid_patch_names) == 0:
            continue

        try:
            features = torch.load(pt_path, map_location='cpu')  # (5, 768)
            if not isinstance(features, torch.Tensor):
                raise ValueError(f"Expected tensor, got {type(features)}")

            if features.shape != (5, 1536):
                print(f"⚠️ Unexpected shape in {pt_file}: {features.shape}")
                continue

            # Directly extend the batch features with slices from the tensor
            batch_features.extend([features[i] for i in range(5)])

            # Generate corresponding patch names
            base = os.path.splitext(pt_file)[0]
            batch_patch_names.extend([f"{base}_{i+1}.png" for i in range(5)])

        except Exception as e:
            print(f"❌ Error reading {pt_path}: {e}")
            continue

        # Run model if batch full or last patch
        last_patch = (idx == len(pt_files) - 1)
        if len(batch_features) >= BATCH_SIZE or last_patch:
            try:
                tensor_batch = torch.stack(batch_features).to(model.device)  # shape: (B, 768)

                model.model.eval()
                with torch.no_grad():
                    outputs = model.model(tensor_batch)
                    predictions = torch.argmax(outputs, dim=1).cpu().tolist()

                updates.extend(zip(batch_patch_names, predictions))

            except Exception as e:
                print(f"❌ Inference error: {e}")

            # Free memory
            del tensor_batch, outputs
            torch.cuda.empty_cache()

            # Reset batch
            batch_features.clear()
            batch_patch_names.clear()

🔍 Processing slides:   0%|          | 0/73 [00:00<?, ?it/s]

1: Processing slide: training_data_01_MSIH


🔍 Processing slides:   1%|▏         | 1/73 [00:53<1:04:15, 53.56s/it]

2: Processing slide: training_data_02_nonMSIH


🔍 Processing slides:   3%|▎         | 2/73 [01:55<1:09:03, 58.36s/it]

...

🔍 Processing slides:  97%|█████████▋| 71/73 [1:22:56<01:56, 58.06s/it]

72: Processing slide: validation_data_30_nonMSIH


🔍 Processing slides:  99%|█████████▊| 72/73 [1:23:20<00:48, 48.07s/it]

73: Processing slide: validation_data_31_nonMSIH


🔍 Processing slides: 100%|██████████| 73/73 [1:23:50<00:00, 68.91s/it]


## Post Processing

In [11]:
update_df = pd.DataFrame(updates, columns=['updated_patchName', 'tissue_label'])
update_df

,updated_patchName,tissue_label
0,training_data_01_MSIH_x15377_y3408_patch00078_...,0
1,training_data_01_MSIH_x15377_y3408_patch00078_...,0
2,training_data_01_MSIH_x15377_y3408_patch00078_...,0
3,training_data_01_MSIH_x15377_y3408_patch00078_...,0
4,training_data_01_MSIH_x15377_y3408_patch00078_...,0
...,...,...
1356605,validation_data_31_nonMSIH_x9872_y9968_patch01...,5
1356606,validation_data_31_nonMSIH_x9872_y9968_patch01...,5
1356607,validation_data_31_nonMSIH_x9872_y9968_patch01...,5
1356608,validation_data_31_nonMSIH_x9872_y9968_patch01...,5


In [12]:
len(set(update_df["updated_patchName"]))

1356610

In [13]:
# update_df = pd.DataFrame(updates, columns=['updated_patchName', 'tissue_label'])
df_expanded.set_index('updated_patchName', inplace=True)
df_expanded.update(update_df.set_index('updated_patchName'))
df_expanded.reset_index(inplace=True)

In [14]:
df_expanded = df_expanded.merge(update_df, on='updated_patchName', how='inner')

# Save the updated CSV if needed
df_expanded.to_csv("./PAIP_Result/hoptimus_labeled_df.csv", index=False)

In [15]:
df_expanded

,updated_patchName,patch_location,patch_name,patch_number,patch_x,patch_y,avg_R,avg_G,avg_B,std_R,std_G,std_B,label,tissue_label
0,training_data_01_MSIH_x20480_y43752_patch00024...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,training_data_01_MSIH_x20480_y43752_patch00024...,24,20480,43752,218.19,221.26,229.02,44.36,39.58,29.21,0,0
1,training_data_01_MSIH_x20480_y43752_patch00024...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,training_data_01_MSIH_x20480_y43752_patch00024...,24,20480,43752,218.19,221.26,229.02,44.36,39.58,29.21,0,0
2,training_data_01_MSIH_x20480_y43752_patch00024...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,training_data_01_MSIH_x20480_y43752_patch00024...,24,20480,43752,218.19,221.26,229.02,44.36,39.58,29.21,0,0
3,training_data_01_MSIH_x20480_y43752_patch00024...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,training_data_01_MSIH_x20480_y43752_patch00024...,24,20480,43752,218.19,221.26,229.02,44.36,39.58,29.21,0,0
4,training_data_01_MSIH_x20480_y43752_patch00024...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,training_data_01_MSIH_x20480_y43752_patch00024...,24,20480,43752,218.19,221.26,229.02,44.36,39.58,29.21,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1356605,validation_data_31_nonMSIH_x48784_y16624_patch...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,validation_data_31_nonMSIH_x48784_y16624_patch...,4822,48784,16624,222.29,210.55,219.74,33.44,39.39,32.08,0,7
1356606,validation_data_31_nonMSIH_x48784_y16624_patch...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,validation_data_31_nonMSIH_x48784_y16624_patch...,4822,48784,16624,222.29,210.55,219.74,33.44,39.39,32.08,0,7
1356607,validation_data_31_nonMSIH_x48784_y16624_patch...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,validation_data_31_nonMSIH_x48784_y16624_patch...,4822,48784,16624,222.29,210.55,219.74,33.44,39.39,32.08,0,7
1356608,validation_data_31_nonMSIH_x48784_y16624_patch...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,validation_data_31_nonMSIH_x48784_y16624_patch...,4822,48784,16624,222.29,210.55,219.74,33.44,39.39,32.08,0,7


## Slide Aggregation

In [16]:
non_white_df= df_expanded.copy()

In [17]:
# non_white_df['patch_name'] = non_white_df['patch_name'].str.replace('.png', '.pt', regex=False)
import re
non_white_df['patch_name'] = non_white_df['patch_name'].str.replace('.png', '.pt', regex=False)
# non_white_df['patch_name'] = non_white_df['patch_name'].str.replace(r'_patch\d{5}', '', regex=True)

In [18]:
non_white_df

,updated_patchName,patch_location,patch_name,patch_number,patch_x,patch_y,avg_R,avg_G,avg_B,std_R,std_G,std_B,label,tissue_label
0,training_data_01_MSIH_x20480_y43752_patch00024...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,training_data_01_MSIH_x20480_y43752_patch00024.pt,24,20480,43752,218.19,221.26,229.02,44.36,39.58,29.21,0,0
1,training_data_01_MSIH_x20480_y43752_patch00024...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,training_data_01_MSIH_x20480_y43752_patch00024.pt,24,20480,43752,218.19,221.26,229.02,44.36,39.58,29.21,0,0
2,training_data_01_MSIH_x20480_y43752_patch00024...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,training_data_01_MSIH_x20480_y43752_patch00024.pt,24,20480,43752,218.19,221.26,229.02,44.36,39.58,29.21,0,0
3,training_data_01_MSIH_x20480_y43752_patch00024...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,training_data_01_MSIH_x20480_y43752_patch00024.pt,24,20480,43752,218.19,221.26,229.02,44.36,39.58,29.21,0,0
4,training_data_01_MSIH_x20480_y43752_patch00024...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,training_data_01_MSIH_x20480_y43752_patch00024.pt,24,20480,43752,218.19,221.26,229.02,44.36,39.58,29.21,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1356605,validation_data_31_nonMSIH_x48784_y16624_patch...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,validation_data_31_nonMSIH_x48784_y16624_patch...,4822,48784,16624,222.29,210.55,219.74,33.44,39.39,32.08,0,7
1356606,validation_data_31_nonMSIH_x48784_y16624_patch...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,validation_data_31_nonMSIH_x48784_y16624_patch...,4822,48784,16624,222.29,210.55,219.74,33.44,39.39,32.08,0,7
1356607,validation_data_31_nonMSIH_x48784_y16624_patch...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,validation_data_31_nonMSIH_x48784_y16624_patch...,4822,48784,16624,222.29,210.55,219.74,33.44,39.39,32.08,0,7
1356608,validation_data_31_nonMSIH_x48784_y16624_patch...,D:\Aamir Gulzar\KSA_project2\paip_data\patch_d...,validation_data_31_nonMSIH_x48784_y16624_patch...,4822,48784,16624,222.29,210.55,219.74,33.44,39.39,32.08,0,7


## Slide aggregation pipeline

In [19]:
import os
import torch
import numpy as np
from tqdm import tqdm
import pandas as pd

# ---------------- CONFIG ----------------
num_classes = 9
# ----------------------------------------

os.makedirs(saving_path, exist_ok=True)

# Load label CSV once. Expect columns 'patch_name' and 'tissue_label'.
# We'll create two maps:
#   1) split_name_map: exact filename without extension -> label  (for cases non_white has _1.._5)
#   2) base_name_map: base patch name (without _1.._5) -> label (fallback)
non_white_df['patch_name'] = non_white_df['patch_name'].astype(str)

split_name_map = {}  # exact match
base_name_map = {}   # fallback by base (without last _idx)
for pn, label in zip(non_white_df['patch_name'], non_white_df['tissue_label']):
    name = os.path.splitext(pn)[0]
    split_name_map[name] = label
    # derive base by removing trailing _<num> if present
    parts = name.rsplit('_', 1)
    if len(parts) == 2 and parts[1].isdigit():
        base = parts[0]
    else:
        base = name
    # only set if not set, keep first occurrence
    if base not in base_name_map:
        base_name_map[base] = label

# Process each slide folder
for slide_name in tqdm(os.listdir(root_dir), desc="Slides"):
    slide_path = os.path.join(root_dir, slide_name)
    if not os.path.isdir(slide_path):
        continue

    files = [f for f in os.listdir(slide_path) if f.endswith('.pt')]
    if not files:
        continue

    # We'll collect features and labels in two lists, then convert to numpy arrays
    features_list = []
    labels_list = []

    for fname in files:
        pt_path = os.path.join(slide_path, fname)
        # load to CPU
        try:
            t = torch.load(pt_path, map_location="cpu")
        except Exception as e:
            print(f"ERROR loading {pt_path}: {e}")
            continue

        # normalize tensor to 2D: (num_crops, D)
        if isinstance(t, torch.Tensor):
            t = t.detach().cpu()
            t = t.squeeze()
            if t.ndim == 1:
                crops = t.unsqueeze(0).numpy()  # (1, D)
            elif t.ndim == 2:
                crops = t.numpy()               # (N, D) e.g., (5, 768)
            else:
                # flatten any extra dims into second dim (rare)
                crops = t.view(t.shape[0], -1).numpy()
        else:
            # skip non-tensor objects
            print(f"Skipping (not tensor): {pt_path}")
            continue

        base_name = os.path.splitext(fname)[0]  # original base without extension

        # For each crop of this patch, create split name and determine label (fast, without saving)
        # We avoid any heavy string ops in loops; simple concatenation and map lookups only.
        for idx in range(crops.shape[0]):
            split_name = f"{base_name}_{idx+1}"  # like file_1, file_2, ...
            features_list.append(crops[idx])     # append numpy 1D array
            # label resolution: exact split name -> fallback to base map
            lbl = split_name_map.get(split_name)
            if lbl is None:
                # fallback: try base name (original patch name without _idx)
                # base_candidate: remove trailing _<digit> if any
                parts = base_name.rsplit('_', 1)
                if len(parts) == 2 and parts[1].isdigit():
                    base_candidate = parts[0]
                else:
                    base_candidate = base_name
                lbl = base_name_map.get(base_candidate, np.nan)
            labels_list.append(lbl)

    # If nothing collected skip
    if not features_list:
        continue

    # Convert lists to arrays
    feats = np.stack(features_list, axis=0).astype(np.float32)  # shape (N, D)
    labels = np.array(labels_list, dtype=np.float32)            # shape (N,)

    # Compute feature dimension
    D = feats.shape[1]

    # Compute group averages per class (0..num_classes-1)
    group_avgs = np.zeros((num_classes, D), dtype=np.float32)

    # Use vectorized grouping: for each label, mask and mean
    # Slight optimization: if labels array contains NaNs, comparisons will be False
    for cls in range(num_classes):
        mask = (labels == cls)
        if mask.any():
            group_avgs[cls] = feats[mask].mean(axis=0)
        else:
            # remains zeros
            pass

    final_tensor = torch.from_numpy(group_avgs)  # shape (num_classes, D)

    # Save the aggregated slide-level tensor
    save_path = os.path.join(saving_path, f"{slide_name}.pt")
    torch.save(final_tensor, save_path)

Slides: 100%|██████████| 73/73 [00:51<00:00,  1.40it/s]


In [ ]:
# import torch

# # Path to your .pt file
# pt_path = r"D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\2-Averaging_FiveCrop\TCGA-AA-3678_nonMSIH.pt"

# # Load to CPU to avoid GPU dependency
# tensor = torch.load(pt_path, map_location="cpu")

# # If it's a tensor
# if isinstance(tensor, torch.Tensor):
#     print("Shape:", tensor.shape)
#     print("Values:\n", tensor)
# else:
#     print("Loaded object type:", type(tensor))
#     print("Value:", tensor)
